In [ ]:
from datasets import load_dataset, concatenate_datasets
from transformers import AutoTokenizer
from transformers import (
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq
)
import torch
import evaluate
import os
import wandb
from datasets import load_from_disk
import pandas as pd
os.environ["WANDB_API_KEY"]="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"

In [17]:
dataset_qa = load_dataset("Open-Orca/OpenOrca", split="train[:1000000]")

In [18]:
dataset_summ = load_dataset("abisee/cnn_dailymail", '1.0.0')

In [16]:
print(dataset_qa.features)
print(dataset_summ['train'].features)

{'id': Value('string'), 'system_prompt': Value('string'), 'question': Value('string'), 'response': Value('string')}
{'article': Value('string'), 'highlights': Value('string'), 'id': Value('string')}


# Tokenization

In [20]:
import numpy as np

sample = dataset_qa.select(range(100000))

qa_q_lens = [len(x.split()) for x in sample['question']]
qa_r_lens = [len(x.split()) for x in sample['response']]
summ_a_lens = [len(x.split()) for x in dataset_summ['train']['article']]
summ_h_lens = [len(x.split()) for x in dataset_summ['train']['highlights']]

print("=== QA (question) ===")
print(f"  mean: {np.mean(qa_q_lens):.0f}")
print(f"  50p:  {np.percentile(qa_q_lens, 50):.0f}")
print(f"  90p:  {np.percentile(qa_q_lens, 90):.0f}")
print(f"  95p:  {np.percentile(qa_q_lens, 95):.0f}")
print(f"  max:  {np.max(qa_q_lens)}")

print("\n=== QA (response) ===")
print(f"  mean: {np.mean(qa_r_lens):.0f}")
print(f"  50p:  {np.percentile(qa_r_lens, 50):.0f}")
print(f"  90p:  {np.percentile(qa_r_lens, 90):.0f}")
print(f"  95p:  {np.percentile(qa_r_lens, 95):.0f}")
print(f"  max:  {np.max(qa_r_lens)}")

print("\n=== SUMMARIZATION (article) ===")
print(f"  mean: {np.mean(summ_a_lens):.0f}")
print(f"  50p:  {np.percentile(summ_a_lens, 50):.0f}")
print(f"  90p:  {np.percentile(summ_a_lens, 90):.0f}")
print(f"  95p:  {np.percentile(summ_a_lens, 95):.0f}")
print(f"  max:  {np.max(summ_a_lens)}")

print("\n=== SUMMARIZATION (highlights) ===")
print(f"  mean: {np.mean(summ_h_lens):.0f}")
print(f"  50p:  {np.percentile(summ_h_lens, 50):.0f}")
print(f"  90p:  {np.percentile(summ_h_lens, 90):.0f}")
print(f"  95p:  {np.percentile(summ_h_lens, 95):.0f}")
print(f"  max:  {np.max(summ_h_lens)}")

=== QA (question) ===
  mean: 158
  50p:  56
  90p:  375
  95p:  466
  max:  6204

=== QA (response) ===
  mean: 114
  50p:  74
  90p:  286
  95p:  387
  max:  840

=== SUMMARIZATION (article) ===
  mean: 692
  50p:  632
  90p:  1166
  95p:  1363
  max:  2347

=== SUMMARIZATION (highlights) ===
  mean: 52
  50p:  48
  90p:  77
  95p:  90
  max:  1296


In [3]:
model_name = "facebook/bart-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [4]:
MAX_INPUT = 512
MAX_TARGET_SUMM = 128
MAX_TARGET_QA = 256

In [15]:
def preprocess_summ(batch):
    inputs = ["summarize: " + art for art in batch['article']]
    targets = batch['highlights']
    
    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT,
        truncation=True,
        padding="max_length"
    )
    labels = tokenizer(
        targets,
        max_length=MAX_TARGET_SUMM,
        truncation=True,
        padding="max_length"
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

def preprocess_qa(batch):
    inputs = ["respond: " + q for q in batch['question']]
    targets = batch['response']
    
    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT,
        truncation=True,
        padding="max_length"
    )
    labels = tokenizer(
        targets,
        max_length=MAX_TARGET_QA,
        truncation=True,
        padding="max_length"
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [19]:
summ_train_tok = dataset_summ['train'].map(
    preprocess_summ,
    batched=True,
    remove_columns=dataset_summ['train'].column_names
)
summ_val_tok = dataset_summ['validation'].map(
    preprocess_summ,
    batched=True,
    remove_columns=dataset_summ['validation'].column_names
)

In [41]:
summ_test_tok = dataset_summ['test'].map(
    preprocess_summ,
    batched=True,
    remove_columns=dataset_summ['test'].column_names
)

Map:   0%|          | 0/11490 [00:00<?, ? examples/s]

In [20]:
qa_subset = dataset_qa.select(range(200000))
qa_tok = qa_subset.map(
    preprocess_qa,
    batched=True,
    remove_columns=qa_subset.column_names
)

In [40]:
qa_split = qa_tok.train_test_split(test_size=0.05, seed=42)
qa_temp = qa_tok.train_test_split(test_size=0.1, seed=42)
qa_split = qa_temp['test'].train_test_split(test_size=0.5, seed=42)

qa_train_tok = qa_temp['train']
qa_val_tok   = qa_split['train']
qa_test_tok  = qa_split['test']

## Merge Datasets

In [34]:
combined_train = concatenate_datasets([summ_train_tok, qa_train_tok]).shuffle(seed=42)
combined_val = concatenate_datasets([summ_val_tok, qa_val_tok]).shuffle(seed=42)

In [35]:
print(f"Combined train: {len(combined_train)}")
print(f"Combined val:   {len(combined_val)}")
print(f"Колонки: {combined_train.column_names}")

Combined train: 477113
Combined val:   23368
Колонки: ['input_ids', 'attention_mask', 'labels']


In [37]:
combined_train.set_format("torch")
combined_val.set_format("torch")

sample = combined_train[0]
for k, v in sample.items():
    print(f"  {k}: shape={v.shape}, dtype={v.dtype}")

  input_ids: shape=torch.Size([512]), dtype=torch.int64
  attention_mask: shape=torch.Size([512]), dtype=torch.int64
  labels: shape=torch.Size([128]), dtype=torch.int64


### save

In [69]:
combined_train.save_to_disk("data/combined_train")
combined_val.save_to_disk("data/combined_val")

Saving the dataset (0/4 shards):   0%|          | 0/477113 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/23368 [00:00<?, ? examples/s]

#### load

In [5]:
combined_train = load_from_disk("./data/combined_train")
combined_val = load_from_disk("./data/combined_val")
combined_train.set_format("torch")
combined_val.set_format("torch")

# Evaluate base-line

# load "facebook/bart-base"

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [7]:
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

In [11]:
results_df = pd.DataFrame(columns=[
    'model_name', 'task', 'rouge1', 'rouge2', 'rougeL', 'bleu',
    'avg_pred_len', 'avg_label_len', 'length_ratio'
])

In [12]:
rouge = evaluate.load("rouge")
bleu = evaluate.load("sacrebleu")

In [22]:
import numpy as np

In [13]:
def evaluate_model(model, tokenizer, dataset, task, model_name, num_samples=200, batch_size=256):
    model.eval()
    device = next(model.parameters()).device
    all_preds, all_labels = [], []

    for i in range(0, num_samples, batch_size):
        batch = dataset.select(range(i, min(i + batch_size, num_samples)))
        input_ids = torch.tensor(batch['input_ids']).to(device)
        attention_mask = torch.tensor(batch['attention_mask']).to(device)

        with torch.no_grad():
            outputs = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=128,
                num_beams=4,
            )

        preds = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        labels = [
            [t if t != -100 else tokenizer.pad_token_id for t in l]
            for l in batch['labels']
        ]
        decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
        all_preds.extend(preds)
        all_labels.extend(decoded_labels)

    rouge_scores = rouge.compute(predictions=all_preds, references=all_labels)
    bleu_score = bleu.compute(predictions=all_preds, references=[[l] for l in all_labels])

    avg_pred_len = np.mean([len(p.split()) for p in all_preds])
    avg_label_len = np.mean([len(l.split()) for l in all_labels])
    length_ratio = avg_pred_len / avg_label_len if avg_label_len > 0 else 0

    print(f"{model_name} | {task}")
    print(f"  ROUGE-1: {rouge_scores['rouge1']:.4f}")
    print(f"  ROUGE-2: {rouge_scores['rouge2']:.4f}")
    print(f"  ROUGE-L: {rouge_scores['rougeL']:.4f}")
    print(f"  BLEU:    {bleu_score['score']:.4f}")
    print(f"  avg_pred_len:  {avg_pred_len:.1f}")
    print(f"  avg_label_len: {avg_label_len:.1f}")
    print(f"  length_ratio:  {length_ratio:.2f}")

    print("examples\n")
    for i in range(3):
        print(f"  PRED:  {all_preds[i][:150]}")
        print(f"  LABEL: {all_labels[i][:150]}")
        print()

    global results_df
    new_row = {
        'model_name': model_name,
        'task': task,
        'rouge1': round(rouge_scores['rouge1'], 4),
        'rouge2': round(rouge_scores['rouge2'], 4),
        'rougeL': round(rouge_scores['rougeL'], 4),
        'bleu': round(bleu_score['score'], 4),
        'avg_pred_len': round(avg_pred_len, 1),
        'avg_label_len': round(avg_label_len, 1),
        'length_ratio': round(length_ratio, 2),
    }
    results_df = pd.concat([results_df, pd.DataFrame([new_row])], ignore_index=True)

In [25]:
evaluate_model(model, tokenizer, summ_val_tok, task="summarization", model_name="bart-base-baseline")
evaluate_model(model, tokenizer, qa_val_tok,   task="qa",            model_name="bart-base-baseline")

results_df

bart-base-baseline | summarization
  ROUGE-1: 0.2641
  ROUGE-2: 0.1066
  ROUGE-L: 0.1767
  BLEU:    5.2666
  avg_pred_len:  97.0
  avg_label_len: 33.3
  length_ratio:  2.91
examples

  PRED:  summarize: (CNN)Share, and your gift will be multiplied. That may sound like an esoteric adage, but when Zully Broussard selflessly decided to give on
  LABEL: Zully Broussard decided to give a kidney to a stranger . A new computer program helped her donation spur transplants for six kidney patients .

  PRED:  summarize: (CNN)On the 6th of April 1996, San Jose Clash and DC United strode out in front of 31,683 expectant fans at the Spartan Stadium in San Jose
  LABEL: The 20th MLS season begins this weekend . League has changed dramatically since its inception in 1996 . Some question whether rules regarding salary c

  PRED:  summarize: (CNN)French striker Bafetimbi Gomis, who has a history of fainting, said he is now "feeling well" after collapsing during Swansea's 3-2 los
  LABEL: Bafetimbi Gomi

In [36]:
import gc

gc.collect()
torch.cuda.empty_cache()

In [37]:
print(torch.cuda.memory_allocated() / 1024**2, "MB allocated")
print(torch.cuda.memory_reserved() / 1024**2, "MB reserved")

8.125 MB allocated
20.0 MB reserved


In [26]:
torch.cuda.empty_cache()

In [ ]:
def train_model(
    model_name="facebook/bart-base",
    experiment_name="bart-base-multitask",
    num_train_epochs=3,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    gradient_accumulation_steps=2,
    warmup_steps=500,
    weight_decay=0.01,
    generation_max_length=256,
    logging_steps=200,
    eval_steps=1000,
    save_steps=1000,
):
    
    wandb.init(project="nlp-lab-2", name=experiment_name, reinit=True)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    print(f"Model: {model_name} on {device}")

    data_collator = DataCollatorForSeq2Seq(
        tokenizer,
        model=model,
        padding=True,
        label_pad_token_id=-100
    )

    training_args = Seq2SeqTrainingArguments(
        output_dir=f"./{experiment_name}",
        num_train_epochs=num_train_epochs,
        per_device_train_batch_size=per_device_train_batch_size,
        per_device_eval_batch_size=per_device_eval_batch_size,
        warmup_steps=warmup_steps,
        weight_decay=weight_decay,
        predict_with_generate=True,
        generation_max_length=generation_max_length,
        logging_steps=logging_steps,
        eval_steps=eval_steps,
        save_steps=save_steps,
        eval_strategy="steps",
        save_total_limit=2,
        load_best_model_at_end=True,
        fp16=True,
        gradient_accumulation_steps=gradient_accumulation_steps,
        report_to="wandb",
        run_name=experiment_name,
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=combined_train,
        eval_dataset=combined_val,
        processing_class=tokenizer,
        data_collator=data_collator,
    )

    print(f"steps - {(len(combined_train) // (per_device_train_batch_size * gradient_accumulation_steps)) * num_train_epochs:,}")
    trainer.train()
    wandb.finish()

    return model, trainer

In [39]:
model, trainer = train_model()

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

Model: facebook/bart-base on cuda
steps - 11,181


Step,Training Loss,Validation Loss
1000,2.154946,0.988870
2000,2.077239,0.950270
3000,2.002215,0.931788
4000,1.934547,0.918087
5000,1.933752,0.910060
6000,1.903802,0.903015
7000,1.885336,0.895536
8000,1.847634,0.891270
9000,1.829736,0.888590
10000,1.840769,0.885872


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


eval/loss,█▅▄▃▃▂▂▁▁▁▁
eval/runtime,▅▅▃▆▃▁▂▆█▃▂
eval/samples_per_second,▄▄▆▃▅█▇▃▁▆▇
eval/steps_per_second,▄▄▆▃▅█▇▃▁▆▇
train/epoch,▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇█
train/global_step,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇███
train/grad_norm,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/learning_rate,▄████▇▇▇▇▇▆▆▆▆▆▆▅▅▅▅▅▅▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▁▁
train/loss,█▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
eval/loss,0.88456
eval/runtime,31.2929


In [32]:
torch.cuda.empty_cache()

In [42]:
evaluate_model(model, tokenizer, summ_test_tok, task="summarization", model_name="bart-base-finetuned")
evaluate_model(model, tokenizer, qa_test_tok,   task="qa",            model_name="bart-base-finetuned")


bart-base-finetuned | summarization
  ROUGE-1: 0.3365
  ROUGE-2: 0.1445
  ROUGE-L: 0.2491
  BLEU:    10.8139
  avg_pred_len:  47.1
  avg_label_len: 34.7
  length_ratio:  1.36
examples

  PRED:  The Palestinian Authority officially becomes the 123rd member of the International Criminal Court . The formal accession was marked with a ceremony at
  LABEL: Membership gives the ICC jurisdiction over alleged crimes committed in Palestinian territories since last June . Israel and the United States opposed 

  PRED:  A stray pooch in Washington State has used up at least three of her own . Theia, a friendly white-and-black bully breed mix, was hit by a car and buri
  LABEL: Theia, a bully breed mix, was apparently hit by a car, whacked with a hammer and buried in a field . "She's a true miracle dog and she deserves a good

  PRED:  Iranian foreign minister Mohammad Javad Zarif has been U.S. Secretary of State John Kerry's opposite number in nuclear talks . Zarif was nominated to 
  LABEL: Moha

In [43]:
results_df

,model_name,task,rouge1,rouge2,rougeL,bleu,avg_pred_len,avg_label_len,length_ratio
0,bart-base-baseline,summarization,0.2641,0.1066,0.1767,5.2666,97.0,33.3,2.91
1,bart-base-baseline,summarization,0.2641,0.1066,0.1767,5.2666,97.0,33.3,2.91
2,bart-base-baseline,qa,0.2627,0.1393,0.1975,9.4759,58.4,85.8,0.68
3,bart-base-finetuned,summarization,0.3365,0.1445,0.2491,10.8139,47.1,34.7,1.36
4,bart-base-finetuned,qa,0.3716,0.2285,0.3107,7.9987,39.8,86.0,0.46


# Generation head tuning

In [47]:
def evaluate_decoding_strategies(model, tokenizer, dataset, task, num_samples=200, batch_size=32):
    
    strategies = {
        "greedy": {
            "num_beams": 1,
            "do_sample": False,
        },
        "beam_search_4": {
            "num_beams": 4,
            "do_sample": False,
        },
        "beam_search_8": {
            "num_beams": 8,
            "do_sample": False,
        },
        "top_k_sampling": {
            "num_beams": 1,
            "do_sample": True,
            "top_k": 50,
        },
        "top_p_sampling": {
            "num_beams": 1,
            "do_sample": True,
            "top_p": 0.95,
        },
        
        "temp_low": {
            "num_beams": 1,
            "do_sample": True,
            "temperature": 0.7, 
        },
        "temp_high": {
            "num_beams": 1,
            "do_sample": True,
            "temperature": 1.5, 
        },
        "top_p_temp": {
            "num_beams": 1,
            "do_sample": True,
            "top_p": 0.95,
            "temperature": 0.7,  
        },
    }
    
    for strategy_name, gen_kwargs in strategies.items():
        print(f"Evaluating: {strategy_name}")
        model.eval()
        device = next(model.parameters()).device
        all_preds, all_labels = [], []
        
        for i in range(0, num_samples, batch_size):
            batch = dataset.select(range(i, min(i + batch_size, num_samples)))
            input_ids = torch.tensor(batch['input_ids']).to(device)
            attention_mask = torch.tensor(batch['attention_mask']).to(device)
            
            with torch.no_grad():
                outputs = model.generate(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    max_new_tokens=128,
                    **gen_kwargs
                )
            
            preds = tokenizer.batch_decode(outputs, skip_special_tokens=True)
            labels = [
                [t if t != -100 else tokenizer.pad_token_id for t in l]
                for l in batch['labels']
            ]
            decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
            all_preds.extend(preds)
            all_labels.extend(decoded_labels)
        
        rouge_scores = rouge.compute(predictions=all_preds, references=all_labels)
        bleu_score = bleu.compute(predictions=all_preds, references=[[l] for l in all_labels])
        avg_pred_len = np.mean([len(p.split()) for p in all_preds])
        avg_label_len = np.mean([len(l.split()) for l in all_labels])

        print(f"  ROUGE-1: {rouge_scores['rouge1']:.4f}")
        print(f"  ROUGE-2: {rouge_scores['rouge2']:.4f}")
        print(f"  ROUGE-L: {rouge_scores['rougeL']:.4f}")
        print(f"  BLEU:    {bleu_score['score']:.4f}")
        print(f"  avg_pred_len:  {avg_pred_len:.1f}")
        print(f"  avg_label_len: {avg_label_len:.1f}")
        print(f"  length_ratio:  {avg_pred_len / avg_label_len:.2f}")

        global results_df
        new_row = {
            'model_name': f"bart-finetuned-{strategy_name}",
            'task': task,
            'rouge1': round(rouge_scores['rouge1'], 4),
            'rouge2': round(rouge_scores['rouge2'], 4),
            'rougeL': round(rouge_scores['rougeL'], 4),
            'bleu': round(bleu_score['score'], 4),
            'avg_pred_len': round(avg_pred_len, 1),
            'avg_label_len': round(avg_label_len, 1),
            'length_ratio': round(avg_pred_len / avg_label_len, 2),
        }
        results_df = pd.concat([results_df, pd.DataFrame([new_row])], ignore_index=True)


In [48]:
evaluate_decoding_strategies(model, tokenizer, summ_test_tok, task="summarization")

The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Evaluating: greedy


  ROUGE-1: 0.3215
  ROUGE-2: 0.1294
  ROUGE-L: 0.2412
  BLEU:    10.4242
  avg_pred_len:  41.8
  avg_label_len: 34.7
  length_ratio:  1.20
Evaluating: beam_search_4


  ROUGE-1: 0.3366
  ROUGE-2: 0.1440
  ROUGE-L: 0.2485
  BLEU:    10.8015
  avg_pred_len:  47.0
  avg_label_len: 34.7
  length_ratio:  1.35
Evaluating: beam_search_8


  ROUGE-1: 0.3258
  ROUGE-2: 0.1346
  ROUGE-L: 0.2386
  BLEU:    9.6839
  avg_pred_len:  47.6
  avg_label_len: 34.7
  length_ratio:  1.37
Evaluating: top_k_sampling


  ROUGE-1: 0.2790
  ROUGE-2: 0.0838
  ROUGE-L: 0.1964
  BLEU:    5.9519
  avg_pred_len:  44.5
  avg_label_len: 34.7
  length_ratio:  1.28
Evaluating: top_p_sampling


  ROUGE-1: 0.2936
  ROUGE-2: 0.0913
  ROUGE-L: 0.2038
  BLEU:    7.0445
  avg_pred_len:  43.9
  avg_label_len: 34.7
  length_ratio:  1.26
Evaluating: temp_low


  ROUGE-1: 0.3045
  ROUGE-2: 0.1121
  ROUGE-L: 0.2182
  BLEU:    8.4965
  avg_pred_len:  42.4
  avg_label_len: 34.7
  length_ratio:  1.22
Evaluating: temp_high


  ROUGE-1: 0.2238
  ROUGE-2: 0.0443
  ROUGE-L: 0.1493
  BLEU:    3.0264
  avg_pred_len:  50.0
  avg_label_len: 34.7
  length_ratio:  1.44
Evaluating: top_p_temp


  ROUGE-1: 0.3106
  ROUGE-2: 0.1087
  ROUGE-L: 0.2201
  BLEU:    8.1166
  avg_pred_len:  42.5
  avg_label_len: 34.7
  length_ratio:  1.22


In [46]:
wandb.init(project="nlp-assignment-2", name="bart-base-multitask", resume="allow")
artifact = wandb.Artifact(name="bart-base-multitask", type="model")
artifact.add_dir("bart-base-multitask")
wandb.log_artifact(artifact)
wandb.finish()

wandb: Adding directory to artifact (bart-base-multitask)... Done. 4.2s


In [49]:
results_df

,model_name,task,rouge1,rouge2,rougeL,bleu,avg_pred_len,avg_label_len,length_ratio
0,bart-base-baseline,summarization,0.2641,0.1066,0.1767,5.2666,97.0,33.3,2.91
1,bart-base-baseline,summarization,0.2641,0.1066,0.1767,5.2666,97.0,33.3,2.91
2,bart-base-baseline,qa,0.2627,0.1393,0.1975,9.4759,58.4,85.8,0.68
3,bart-base-finetuned,summarization,0.3365,0.1445,0.2491,10.8139,47.1,34.7,1.36
4,bart-base-finetuned,qa,0.3716,0.2285,0.3107,7.9987,39.8,86.0,0.46
5,bart-finetuned-greedy,summarization,0.3215,0.1294,0.2412,10.4242,41.8,34.7,1.2
6,bart-finetuned-beam_search_4,summarization,0.3366,0.144,0.2485,10.8015,47.0,34.7,1.35
7,bart-finetuned-beam_search_8,summarization,0.3258,0.1346,0.2386,9.6839,47.6,34.7,1.37
8,bart-finetuned-top_k_sampling,summarization,0.279,0.0838,0.1964,5.9519,44.5,34.7,1.28
9,bart-finetuned-top_p_sampling,summarization,0.2936,0.0913,0.2038,7.0445,43.9,34.7,1.26
